In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP05 - Contingent Workers in High-Risk Jurisdictions
   
   BUSINESS QUESTION: 
   Are there contingent workers associated with the Assessable Unit operating in 
   high-risk jurisdictions?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGH-RISK JURISDICTIONS: Queries the td_country_risk_rating_abac table to isolate 
      countries where the FinalABACRating is strictly 'High'.
   3. CONTINGENT WORKERS (SMART PADDING ADDED): Filters the HR Enterprise List to find 
      employees where WorkerType contains the word 'contingent'. Applies the 
      "Smart Padding" rule (pads leading zeros ONLY if the ID is pure numbers and 
      strictly under 4 digits) to perfectly match the CC Bootstrap as a string,
      replacing the old TRY_CAST to INT that was dropping alphanumeric IDs.
   4. FILTERING & MAPPING: INNER JOINS the contingent workers to the High-Risk table 
      to strictly keep high-risk workers. Then maps them to AU_IDs via the CC Bootstrap.
   5. OUTPUT FORMATTING: Rolls up the data by AU_ID. LEFT JOINS this to the Master AU 
      anchor. If mapped high-risk workers exist, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    -- 2. Grab jurisdictions strictly rated as 'High' risk
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

Contingent_Workers AS (
    -- 3. Filter HR List for Contingent Workers and apply Smart Padding for joining
    SELECT 
        TRIM(CostCenterID) AS Raw_CC,
        CASE 
            WHEN TRIM(CostCenterID) RLIKE '^[0-9]+$' 
            THEN LPAD(RIGHT(TRIM(CostCenterID), 4), 4, '0')
            ELSE UPPER(TRIM(CostCenterID)) 
        END AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

High_Risk_Contingent_Workers AS (
    -- Strictly keep only the contingent workers in high-risk countries
    SELECT 
        c.Cleaned_CC,
        c.Workcountry
    FROM Contingent_Workers c
    INNER JOIN High_Risk_Countries h
        ON c.Workcountry = h.CountryName
    WHERE c.Cleaned_CC IS NOT NULL AND c.Cleaned_CC != ''
),

CC_Mapping AS (
    -- Standardize the CC Mapping table (No INT casting, treated as strings)
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0') ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING))) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND Cost_Center_ID IS NOT NULL
),

Flagged_AUs AS (
    -- 4. Map the high-risk contingent workers to their respective AU_IDs
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM High_Risk_Contingent_Workers hw
    INNER JOIN CC_Mapping m
        ON hw.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 5. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EMP05' AS QuestionID,               
    CASE 
        WHEN COALESCE(f.Flagged_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP05 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final Yes / No
   response was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE UPPER(TRIM(FinalABACRating)) = 'HIGH'
),

Contingent_Workers AS (
    SELECT 
        CASE 
            WHEN TRIM(CostCenterID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenterID), 4), 4, '0')
            ELSE UPPER(TRIM(CostCenterID)) 
        END AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

High_Risk_Contingent_Workers AS (
    SELECT 
        c.Cleaned_CC,
        c.Workcountry
    FROM Contingent_Workers c
    INNER JOIN High_Risk_Countries h
        ON c.Workcountry = h.CountryName
    WHERE c.Cleaned_CC IS NOT NULL AND c.Cleaned_CC != ''
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0') ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING))) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND Cost_Center_ID IS NOT NULL
),

Mapped_High_Risk_Workers AS (
    SELECT 
        m.AU_ID,
        hw.Cleaned_CC,
        hw.Workcountry
    FROM High_Risk_Contingent_Workers hw
    INNER JOIN CC_Mapping m
        ON hw.Cleaned_CC = m.Mapped_CC
),

High_Risk_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Workcountry))) AS High_Risk_Country_List,
        COUNT(*) AS High_Risk_Worker_Count
    FROM Mapped_High_Risk_Workers
    GROUP BY AU_ID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EMP05' AS QuestionID,
    CASE WHEN COALESCE(h.High_Risk_Worker_Count, 0) > 0 THEN 'Yes' ELSE 'No' END AS Response,
    COALESCE(h.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(h.High_Risk_Country_List, 'n/a') AS High_Risk_Country_List,
    COALESCE(h.High_Risk_Worker_Count, 0) AS High_Risk_Worker_Count,
    CASE 
        WHEN COALESCE(h.High_Risk_Worker_Count, 0) > 0 THEN CONCAT('Yes because ', CAST(h.High_Risk_Worker_Count AS STRING), ' high-risk contingent worker record(s) mapped to this AU.')
        ELSE 'No because no high-risk contingent worker records mapped to this AU.'
    END AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN High_Risk_By_AU h
    ON mast.BusinessID = h.AU_ID
ORDER BY mast.BusinessID;
